# 🚀 BLOCO 1 — Carregar modelo BLIP 31 Março de 2026

In [3]:
from transformers import BlipProcessor, BlipForConditionalGeneration
import torch

caminho_modelo = r"C:\Users\berna\GitHub\PROJETO APLICADO XP_2026\notebooks rascunho\meu_modelo_blip_31_03_2026"

processor = BlipProcessor.from_pretrained(caminho_modelo)
model = BlipForConditionalGeneration.from_pretrained(caminho_modelo)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

C:\Users\berna\GitHub\XP DESAFIO FINAL outubro de 2025\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|█████████████████████████████████████████████████████████████| 472/472 [00:00<00:00, 2256.50it/s]
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


BlipForConditionalGeneration(
  (vision_model): BlipVisionModel(
    (embeddings): BlipVisionEmbeddings(
      (patch_embedding): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (encoder): BlipEncoder(
      (layers): ModuleList(
        (0-11): 12 x BlipEncoderLayer(
          (self_attn): BlipAttention(
            (dropout): Dropout(p=0.0, inplace=False)
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (projection): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): BlipMLP(
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        )
      )
    )
    (post_layernorm): LayerNorm((768,), eps=1e-0

# 📊 BLOCO 2 — Carregar e filtrar glossário

In [4]:
import pandas as pd

caminho_glossario = r"C:\Users\berna\GitHub\PROJETO APLICADO XP_2026\Base de dados\planilhas Geradas\glossario_consolidado.csv"

glossario = pd.read_csv(
    caminho_glossario,
    sep=";",                # 👈 separador correto
    encoding="utf-8",
    engine="python",        # 👈 resolve textos com ;
    on_bad_lines="skip"
)

# limpar nomes das colunas
glossario.columns = glossario.columns.str.strip()

print("Colunas:", glossario.columns.tolist())
glossario.head()


# filtro na coluna TERMO
glossario_filtrado = glossario[
    glossario["TERMO"].str.lower().str.contains("exploração|produção", na=False)
]

print(f"\nTotal de termos filtrados: {len(glossario_filtrado)}")

display(glossario_filtrado.head(47))



Colunas: ['TERMO', 'DEFINIÇÃO', 'REFERENCIA']

Total de termos filtrados: 47


,TERMO,DEFINIÇÃO,REFERENCIA
100,Dados Exclusivos (Exploração e Produção),Dados adquiridos por Concessionário nos limite...,glossario_anp
103,Dados Mistos (Exploração e Produção),Dados resultantes do reprocessamento conjunto ...,glossario_anp
104,Dados Não-Exclusivos (Exploração),Dados adquiridos por EAD em área que seja ou n...,glossario_anp
110,Data de Início da Produção,"Data em que ocorrer a primeira medição, em cad...",glossario_anp
148,Etapa da Fase de Produção,"Estágio em que se encontra um campo, ou seja, ...",glossario_anp
149,Etapa de Produção,Período iniciado na data de entrega da declara...,glossario_anp
155,Fase de Exploração,Período de tempo definido para a exploração. P...,glossario_anp
156,Fase de Produção,Período de tempo definido para produção. PORTA...,glossario_anp
195,Individualização da Produção,"Viabilização, por um projeto único, do desenvo...",glossario_anp
211,Lavra ou Produção,Conjunto de operações coordenadas de extração ...,glossario_anp


# 🚀 BLOCO 3 — Processar pasta de imagens + usar modelo + glossário

PASTA DE IMAGENS
        ↓
MODELO BLIP (descrição automática)
        ↓
GLOSSÁRIO FILTRADO
        ↓
ENRIQUECIMENTO SEMÂNTICO
        ↓
TABELA FINAL

In [5]:
# 🚀 BLOCO 3.1 — Listar imagens da pasta
import os

pasta_imagens = r"C:\Users\berna\GitHub\PROJETO APLICADO XP_2026\notebooks rascunho\imagens_5570"

arquivos = [f for f in os.listdir(pasta_imagens) if f.lower().endswith((".jpg", ".png", ".jpeg"))]

print(f"Total de imagens encontradas: {len(arquivos)}")
print("Exemplo de arquivos:", arquivos[:5])

Total de imagens encontradas: 5570
Exemplo de arquivos: ['img_0.jpg', 'img_1.jpg', 'img_10.jpg', 'img_100.jpg', 'img_1000.jpg']


In [6]:
#🧪 TESTE AGORA (importante)
import os

arquivos_modelo = os.listdir(caminho_modelo)
print(arquivos_modelo)

['config.json', 'generation_config.json', 'model.safetensors', 'processor_config.json', 'tokenizer.json', 'tokenizer_config.json']


# 🚀 Próximo passo (agora vai rodar)

Vamos testar a geração de descrição (de novo, mas agora com certeza vai funcionar):

In [7]:
from PIL import Image
import os

arquivo_teste = arquivos[0]
caminho_teste = os.path.join(pasta_imagens, arquivo_teste)

print("Imagem de teste:", arquivo_teste)

image = Image.open(caminho_teste).convert("RGB")

inputs = processor(images=image, return_tensors="pt").to(device)

output = model.generate(
    **inputs,
    max_length=50,
    num_beams=5,
    early_stopping=True
)

descricao = processor.decode(output[0], skip_special_tokens=True)

print("\nDescrição gerada:")
print(descricao)

Imagem de teste: img_0.jpg

Descrição gerada:
homens a bordo do navio - plataforma fpso cidade de sao vicente, operando no campo de tupi na bacia de santos. tags : bacia _ maritima | bacia _ de _ santos


# 🔥 BLOCO 3.5 — Matching por palavras (mais flexível)

In [10]:
def buscar_glossario_melhorado(descricao, glossario_df):
    resultados = []

    descricao_lower = descricao.lower()

    for _, row in glossario_df.iterrows():
        termo = str(row["TERMO"]).lower()

        # quebra o termo em palavras
        palavras_termo = termo.split()

        for palavra in palavras_termo:
            if palavra in descricao_lower:
                resultados.append({
                    "termo": row["TERMO"],
                    "definicao": row["DEFINIÇÃO"]
                })
                break

    return resultados

# 🧪 BLOCO 3.6 — Testar versão melhorada

In [11]:
resultados_glossario = buscar_glossario_melhorado(descricao, glossario_filtrado)

print("Termos encontrados:\n")

for r in resultados_glossario:
    print("-", r["termo"])

Termos encontrados:

- Dados Exclusivos (Exploração e Produção)
- Dados Mistos (Exploração e Produção)
- Data de Início da Produção
- Etapa da Fase de Produção
- Etapa de Produção
- Fase de Exploração
- Fase de Produção
- Individualização da Produção
- Método de Similaridade (Produção)
- Método Volumétrico (Exploração e Produção)
- Pontos de Medição da Produção
- Potencial de Produção do Poço
- Processamento (Exploração e Produção)
- Produção de Biocombustível
- Programa Anual de Produção
- Projeto Piloto de Produção
- Receita Bruta da Produção
- Receita Líquida da Produção
- Reprocessamento (Exploração e Produção)
- Sistema de Produção
- Sistema de Produção Marítimo
- Unidade de Produção Marítima
- Unidade de Produção Terrestre
- Volume de Produção Fiscalizada
- Volume Total da Produção
- Demonstrativo de Produção e Movimentação de Produtos (DPMP)
- Etapa de Produção
- Fase de Exploração
- Fase de Produção
- Individualização da Produção
- Partilha de Produção
- Poço Explotatório de Pr

# 🔥 BLOCO 3.7 — Remover duplicado

In [12]:
def remover_duplicados(resultados):
    vistos = set()
    resultados_unicos = []

    for r in resultados:
        if r["termo"] not in vistos:
            resultados_unicos.append(r)
            vistos.add(r["termo"])

    return resultados_unicos

# 🔥 BLOCO 3.8 — Aplicar limpeza

In [13]:
resultados_unicos = remover_duplicados(resultados_glossario)

print("Termos únicos:\n")

for r in resultados_unicos:
    print("-", r["termo"])

Termos únicos:

- Dados Exclusivos (Exploração e Produção)
- Dados Mistos (Exploração e Produção)
- Data de Início da Produção
- Etapa da Fase de Produção
- Etapa de Produção
- Fase de Exploração
- Fase de Produção
- Individualização da Produção
- Método de Similaridade (Produção)
- Método Volumétrico (Exploração e Produção)
- Pontos de Medição da Produção
- Potencial de Produção do Poço
- Processamento (Exploração e Produção)
- Produção de Biocombustível
- Programa Anual de Produção
- Projeto Piloto de Produção
- Receita Bruta da Produção
- Receita Líquida da Produção
- Reprocessamento (Exploração e Produção)
- Sistema de Produção
- Sistema de Produção Marítimo
- Unidade de Produção Marítima
- Unidade de Produção Terrestre
- Volume de Produção Fiscalizada
- Volume Total da Produção
- Demonstrativo de Produção e Movimentação de Produtos (DPMP)
- Partilha de Produção
- Poço Explotatório de Produção


# BLOCO 3.9 — Limitar resultados (TOP N)

Agora controla o excesso:

In [14]:
top_resultados = resultados_unicos[:5]

print("\nTop termos mais relevantes:\n")

for r in top_resultados:
    print("-", r["termo"])


Top termos mais relevantes:

- Dados Exclusivos (Exploração e Produção)
- Dados Mistos (Exploração e Produção)
- Data de Início da Produção
- Etapa da Fase de Produção
- Etapa de Produção


# 🔥 BLOCO 3.10 — Pontuar relevância

In [15]:
def buscar_glossario_com_score(descricao, glossario_df):
    resultados = []

    descricao_lower = descricao.lower()

    for _, row in glossario_df.iterrows():
        termo = str(row["TERMO"]).lower()
        palavras_termo = termo.split()

        score = 0

        for palavra in palavras_termo:
            if palavra in descricao_lower:
                score += 1

        if score > 0:
            resultados.append({
                "termo": row["TERMO"],
                "definicao": row["DEFINIÇÃO"],
                "score": score
            })

    return resultados

# 🔥 BLOCO 3.11 — Ordenar por relevância

In [16]:
resultados_score = buscar_glossario_com_score(descricao, glossario_filtrado)

# ordenar do maior score para o menor
resultados_ordenados = sorted(resultados_score, key=lambda x: x["score"], reverse=True)

# pegar top 5
top_resultados = resultados_ordenados[:5]

print("Top termos mais relevantes:\n")

for r in top_resultados:
    print(f"- {r['termo']} (score: {r['score']})")

Top termos mais relevantes:

- Demonstrativo de Produção e Movimentação de Produtos (DPMP) (score: 3)
- Data de Início da Produção (score: 2)
- Etapa da Fase de Produção (score: 2)
- Pontos de Medição da Produção (score: 2)
- Potencial de Produção do Poço (score: 2)


In [17]:
#🔥 BLOCO 3.12 — Lista de palavras irrelevantes

palavras_irrelevantes = {
    "produção", "exploração", "dados", "data", "etapa",
    "fase", "processo", "programa", "volume", "sistema"
}

In [18]:
# 🔥 BLOCO 3.13 — Score ignorando palavras genéricas
def buscar_glossario_filtrado(descricao, glossario_df):
    resultados = []

    descricao_lower = descricao.lower()

    for _, row in glossario_df.iterrows():
        termo = str(row["TERMO"]).lower()
        palavras_termo = termo.split()

        score = 0

        for palavra in palavras_termo:
            if palavra in palavras_irrelevantes:
                continue  # ignora palavras genéricas

            if palavra in descricao_lower:
                score += 1

        if score > 0:
            resultados.append({
                "termo": row["TERMO"],
                "definicao": row["DEFINIÇÃO"],
                "score": score
            })

    return resultados

In [19]:
#🔥 BLOCO 3.14 — Testar versão refinada 

resultados = buscar_glossario_filtrado(descricao, glossario_filtrado)

resultados_ordenados = sorted(resultados, key=lambda x: x["score"], reverse=True)

top_resultados = resultados_ordenados[:5]

print("Top termos mais relevantes:\n")

for r in top_resultados:
    print(f"- {r['termo']} (score: {r['score']})")

Top termos mais relevantes:

- Demonstrativo de Produção e Movimentação de Produtos (DPMP) (score: 3)
- Data de Início da Produção (score: 2)
- Etapa da Fase de Produção (score: 2)
- Pontos de Medição da Produção (score: 2)
- Potencial de Produção do Poço (score: 2)


In [20]:
#🚀 BLOCO 3.15 — Extrair palavras-chave da descrição
def extrair_palavras_chave(descricao):
    palavras = descricao.lower().split()

    # remove palavras muito curtas
    palavras = [p for p in palavras if len(p) > 3]

    return palavras

In [21]:
palavras_chave = extrair_palavras_chave(descricao)

print("Palavras-chave da descrição:\n")
print(palavras_chave)

Palavras-chave da descrição:

['homens', 'bordo', 'navio', 'plataforma', 'fpso', 'cidade', 'vicente,', 'operando', 'campo', 'tupi', 'bacia', 'santos.', 'tags', 'bacia', 'maritima', 'bacia', 'santos']


In [22]:
#BLOCO 4.1 — Função completa por imagem
def processar_imagem(caminho_img, glossario_df):
    try:
        image = Image.open(caminho_img).convert("RGB")

        inputs = processor(images=image, return_tensors="pt").to(device)

        output = model.generate(
            **inputs,
            max_length=50,
            num_beams=5,
            early_stopping=True
        )

        descricao = processor.decode(output[0], skip_special_tokens=True)

        # extrair palavras-chave
        palavras = descricao.lower().split()

        # limpar palavras
        import re
        palavras = [re.sub(r"[^\w\s]", "", p) for p in palavras if len(p) > 3]
        palavras = list(set(palavras))

        return {
            "imagem": os.path.basename(caminho_img),
            "descricao": descricao,
            "palavras_chave": palavras
        }

    except Exception as e:
        return {
            "imagem": os.path.basename(caminho_img),
            "erro": str(e)
        }

In [24]:
#🚀 BLOCO 4.2 (AJUSTADO) — Rodar só 100 imagens
from tqdm import tqdm

resultados = []

# pegar apenas as primeiras 100 imagens
arquivos_teste = arquivos[:100]

for arquivo in tqdm(arquivos_teste):
    caminho_img = os.path.join(pasta_imagens, arquivo)

    resultado = processar_imagem(caminho_img, glossario_filtrado)
    resultados.append(resultado)

100%|████████████████████████████████████████████████████████████████████████████████| 100/100 [20:51<00:00, 12.51s/it]


In [29]:
🔥 BLOCO 4.3 — Transformar em DataFrame
from tqdm import tqdm
import pandas as pd
import os

resultados = []

# pegar apenas as primeiras 100 imagens
arquivos_teste = arquivos[:100]

for arquivo in tqdm(arquivos_teste):
    caminho_img = os.path.join(pasta_imagens, arquivo)

    resultado = processar_imagem(caminho_img, glossario_filtrado)
    resultados.append(resultado)

# transformar em DataFrame
df_resultados = pd.DataFrame(resultados)

# visualizar
df_resultados.head()

SyntaxError: invalid character '🔥' (U+1F525) (1767472344.py, line 1)

In [30]:
df_resultados.to_csv("resultado_100_imagens.csv", sep=";", index=False)

NameError: name 'df_resultados' is not defined

#🔥 BLOCO 4.2 — Rodar em TODAS as imagens
from tqdm import tqdm

resultados = []

for arquivo in tqdm(arquivos):
    caminho_img = os.path.join(pasta_imagens, arquivo)

    resultado = processar_imagem(caminho_img, glossario_filtrado)
    resultados.append(resultado)